In [1]:
# Instalar previamente:
# pip install numpy pandas scikit-learn

import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# -------------------------------------------------
# 1. Cargar los datos del censo
# -------------------------------------------------

datos = fetch_openml(
    name="adult",
    version=2,
    as_frame=True
)

X = datos.data
y = datos.target

# Seleccionar una muestra para facilitar la ejecucion
muestra = X.sample(n=10000, random_state=42).index
X = X.loc[muestra]
y = y.loc[muestra]

# -------------------------------------------------
# 2. Identificar tipos de variables
# -------------------------------------------------

variables_numericas = X.select_dtypes(
    include=["number"]
).columns.tolist()

variables_categoricas = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Preparar las variables numericas
transformacion_numerica = Pipeline([
    ("imputacion", SimpleImputer(strategy="median")),
    ("escalado", StandardScaler())
])

# Preparar las variables categoricas
transformacion_categorica = Pipeline([
    ("imputacion", SimpleImputer(strategy="most_frequent")),
    ("codificacion", OneHotEncoder(handle_unknown="ignore"))
])

# Aplicar el procesamiento segun el tipo de variable
preprocesamiento = ColumnTransformer([
    ("numericas", transformacion_numerica, variables_numericas),
    ("categoricas", transformacion_categorica, variables_categoricas)
])

# -------------------------------------------------
# 3. Separar entrenamiento y prueba
# -------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# -------------------------------------------------
# 4. Crear y entrenar KNN
# -------------------------------------------------

modelo_knn = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("clasificador", KNeighborsClassifier(n_neighbors=5))
])

modelo_knn.fit(X_train, y_train)

# -------------------------------------------------
# 5. Realizar predicciones
# -------------------------------------------------

predicciones = modelo_knn.predict(X_test)

print("Exactitud del modelo KNN:",
      accuracy_score(y_test, predicciones))

print("\nReporte de clasificacion:")
print(classification_report(y_test, predicciones))

# Ejemplo: clasificar una persona del conjunto de prueba
ejemplo = X_test.iloc[[0]]
prediccion = modelo_knn.predict(ejemplo)

print("Caracteristicas del ejemplo:\n", ejemplo)
print("Clase predicha:", prediccion[0])
print("Clase real:", y_test.iloc[0])

Exactitud del modelo KNN: 0.8385

Reporte de clasificacion:
              precision    recall  f1-score   support

       <=50K       0.87      0.92      0.90      1532
        >50K       0.69      0.56      0.62       468

    accuracy                           0.84      2000
   macro avg       0.78      0.74      0.76      2000
weighted avg       0.83      0.84      0.83      2000

Caracteristicas del ejemplo:
        age workclass  fnlwgt education  education-num marital-status  \
15845   23   Private  349156   HS-grad              9  Never-married   

            occupation   relationship   race   sex  capital-gain  \
15845  Farming-fishing  Not-in-family  White  Male             0   

       capital-loss  hours-per-week native-country  
15845             0              45  United-States  
Clase predicha: <=50K
Clase real: <=50K
